In [5]:
"""Map Eqasim public-transport trips to the corresponding SUMO line IDs.
The script combines three data sources:
- an Eqasim CSV containing public-transport legs;
- a SUMO XML file generated from GTFS data;
- the original GTFS archive.

It first tries a direct identifier match. When that is not possible, it compares
GTFS route metadata and stop sequences to find a reliable fallback match. The
script exports a compact mapping file, a detailed diagnostic file, and a file
containing unmatched records.

The input files are read only and are never modified.
"""

import math
import re
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import Path
import pandas as pd


# ============================================================
# INPUT AND OUTPUT PATHS
# ============================================================
# Eqasim public-transport legs to be mapped.
EQASIM_CSV_PATH = Path(r"../eqasim_output_filtered/eqasim_pt_filtered.csv")
# SUMO vehicles and line IDs created during the GTFS-to-SUMO conversion.
SUMO_PT_XML_PATH = Path(r"../3-public_transport/gtfs_pt_vehicles.add.xml")
# Original GTFS feed used to compare routes, directions, shapes, and stops.
GTFS_ZIP_PATH = Path(r"../3-public_transport/input/ca_la_rochelle-aggregated-gtfs.zip")


# The compact file is sufficient for normal pipeline execution.
OUTPUT_MINIMAL_CSV = "output_tools/eqasim_pt_sumo_line_mapping.csv"
# The two additional files make the matching process auditable.
OUTPUT_DEBUG_CSV = "output_tools/eqasim_pt_sumo_line_mapping_debug.csv"
OUTPUT_UNMATCHED_CSV = "output_tools/eqasim_pt_sumo_line_mapping_unmatched.csv"


# ============================================================
# SMALL UTILITY FUNCTIONS
# ============================================================
def normalize_text(value):
    """Return a clean lowercase string, or ``None`` for an empty value."""
    if value is None:
        return None
    if isinstance(value, float) and math.isnan(value):
        return None
    value = str(value).strip()
    if value == "":
        return None
    return value.lower()


def extract_line_suffix(transit_line_id):
    """
    Extract ``X`` from an identifier such as ``CA_LA_ROCHELLE:Line:X``.
    """
    if pd.isna(transit_line_id):
        return None

    transit_line_id = str(transit_line_id)
    match = re.search(r"CA_LA_ROCHELLE:Line:(.+)$", transit_line_id)
    if match:
        return match.group(1)

    # Generic fallback for identifiers that do not use the expected prefix.
    return transit_line_id.split(":")[-1]


def safe_get(dct, key, default=None):
    """Read a dictionary value without failing when the dictionary is missing."""
    if dct is None:
        return default
    return dct.get(key, default)


# ============================================================
# READ THE EQASIM PUBLIC-TRANSPORT LEGS
# ============================================================
def load_eqasim_csv(eqasim_csv_path):
    """Load the Eqasim CSV and verify the columns required for matching."""
    df = pd.read_csv(eqasim_csv_path, sep=";")

    required_columns = [
        "person_id",
        "person_trip_id",
        "leg_index",
        "transit_line_id",
        "transit_route_id",
    ]

    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(
            f"Missing required columns in {eqasim_csv_path}: {missing}"
        )

    return df


# ============================================================
# READ THE SUMO PUBLIC-TRANSPORT VEHICLES
# ============================================================
def load_sumo_pt_xml(sumo_xml_path):
    """
    Parse the SUMO vehicle XML once and build fast lookup dictionaries:
    - direct_vehicle_lookup: vehicle_id -> metadata
    - line_to_meta: line -> representative metadata
    - route_name_to_lines: gtfs.route_name -> set(line)
    """
    direct_vehicle_lookup = {}
    line_to_meta = {}
    route_name_to_lines = defaultdict(set)

    for _, elem in ET.iterparse(sumo_xml_path, events=("end",)):
        if elem.tag != "vehicle":
            continue

        vehicle_id = elem.attrib.get("id")
        line_attr = elem.attrib.get("line")

        params = {}
        for child in list(elem):
            if child.tag == "param":
                key = child.attrib.get("key")
                value = child.attrib.get("value")
                params[key] = value

        gtfs_route_name = params.get("gtfs.route_name")
        gtfs_trip_headsign = params.get("gtfs.trip_headsign")

        direct_vehicle_lookup[vehicle_id] = {
            "vehicle_id": vehicle_id,
            "line": line_attr,
            "gtfs.route_name": gtfs_route_name,
            "gtfs.trip_headsign": gtfs_trip_headsign,
        }

        # One representative metadata record per SUMO "line"
        if line_attr is not None and line_attr not in line_to_meta:
            line_to_meta[line_attr] = {
                "line": line_attr,
                "sample_vehicle_id": vehicle_id,
                "gtfs.route_name": gtfs_route_name,
                "gtfs.trip_headsign": gtfs_trip_headsign,
            }

        if gtfs_route_name is not None and line_attr is not None:
            route_name_to_lines[str(gtfs_route_name)].add(line_attr)

        elem.clear()

    route_name_to_lines = {
        route_name: sorted(lines)
        for route_name, lines in route_name_to_lines.items()
    }

    return direct_vehicle_lookup, line_to_meta, route_name_to_lines


# ============================================================
# READ THE ORIGINAL GTFS DATA
# ============================================================
def load_gtfs(gtfs_zip_path):
    """
    Load the GTFS route, trip, and stop-time tables, then build:
    - gtfs_trip_lookup: trip_id -> metadata
    - stop_seq_lookup: trip_id -> tuple(stop_id, ...)
    """
    with zipfile.ZipFile(gtfs_zip_path) as zf:
        with zf.open("routes.txt") as f:
            routes = pd.read_csv(f, dtype=str)

        with zf.open("trips.txt") as f:
            trips = pd.read_csv(f, dtype=str)

        with zf.open("stop_times.txt") as f:
            stop_times = pd.read_csv(f, dtype=str)

    # route_id -> route_short_name
    route_short_name_by_route_id = routes.set_index("route_id")["route_short_name"].to_dict()

    trips = trips.copy()
    trips["route_short_name"] = trips["route_id"].map(route_short_name_by_route_id)

    gtfs_trip_lookup = trips.set_index("trip_id")[
        ["route_id", "route_short_name", "trip_headsign", "direction_id", "shape_id"]
    ].to_dict(orient="index")

    stop_times = stop_times[["trip_id", "stop_sequence", "stop_id"]].copy()
    stop_times["stop_sequence"] = pd.to_numeric(stop_times["stop_sequence"], errors="coerce")
    stop_times = stop_times.sort_values(["trip_id", "stop_sequence"])

    stop_seq_lookup = stop_times.groupby("trip_id")["stop_id"].apply(tuple).to_dict()

    return gtfs_trip_lookup, stop_seq_lookup


# ============================================================
# SCORE FALLBACK CANDIDATES
# ============================================================
def score_candidate(target_trip_id, candidate_line_trip_id, gtfs_trip_lookup, stop_seq_lookup):
    """
    Compare one Eqasim route with one possible SUMO line.
    The SUMO ``line`` value is assumed to correspond to a GTFS ``trip_id``.

    We use:
    - same route_id
    - same direction_id
    - same trip_headsign
    - same shape_id
    - exact same stop sequence
    - same first/last stop

    Returns
    -------
    score : int
    details : dict
    """
    target = gtfs_trip_lookup.get(target_trip_id)
    candidate = gtfs_trip_lookup.get(candidate_line_trip_id)

    if target is None or candidate is None:
        return -1, {
            "strong_match": False,
            "reasons": ["missing_gtfs_trip"],
        }

    # A larger score means that the candidate is more similar to the target trip.
    score = 0
    reasons = []

    same_route_id = (
        normalize_text(target.get("route_id"))
        == normalize_text(candidate.get("route_id"))
    )
    same_direction_id = (
        normalize_text(target.get("direction_id"))
        == normalize_text(candidate.get("direction_id"))
    )
    same_trip_headsign = (
        normalize_text(target.get("trip_headsign"))
        == normalize_text(candidate.get("trip_headsign"))
    )

    target_shape = normalize_text(target.get("shape_id"))
    candidate_shape = normalize_text(candidate.get("shape_id"))
    same_shape_id = (
        target_shape is not None
        and candidate_shape is not None
        and target_shape == candidate_shape
    )

    target_stops = stop_seq_lookup.get(target_trip_id)
    candidate_stops = stop_seq_lookup.get(candidate_line_trip_id)

    same_stop_sequence_exact = (
        target_stops is not None
        and candidate_stops is not None
        and target_stops == candidate_stops
    )

    same_first_stop = False
    same_last_stop = False
    same_first_last_stops = False

    if target_stops and candidate_stops:
        same_first_stop = target_stops[0] == candidate_stops[0]
        same_last_stop = target_stops[-1] == candidate_stops[-1]
        same_first_last_stops = same_first_stop and same_last_stop

    if same_route_id:
        score += 10
        reasons.append("same_route_id")

    if same_direction_id:
        score += 100
        reasons.append("same_direction_id")

    if same_trip_headsign:
        score += 50
        reasons.append("same_trip_headsign")

    if same_shape_id:
        score += 1000
        reasons.append("same_shape_id")

    if same_stop_sequence_exact:
        score += 2000
        reasons.append("same_stop_sequence_exact")

    if same_first_stop:
        score += 20
        reasons.append("same_first_stop")

    if same_last_stop:
        score += 20
        reasons.append("same_last_stop")

    if same_first_last_stops:
        score += 100
        reasons.append("same_first_last_stops")

    # A high numeric score alone is not enough: at least one reliable structural
    # condition must also be satisfied before a fallback is accepted.
    strong_match = (
        same_stop_sequence_exact
        or same_shape_id
        or (same_direction_id and same_trip_headsign and same_first_last_stops)
    )

    details = {
        "strong_match": strong_match,
        "reasons": reasons,
        "target_route_id": target.get("route_id"),
        "target_route_short_name": target.get("route_short_name"),
        "target_headsign": target.get("trip_headsign"),
        "target_direction_id": target.get("direction_id"),
        "target_shape_id": target.get("shape_id"),
        "candidate_route_id": candidate.get("route_id"),
        "candidate_route_short_name": candidate.get("route_short_name"),
        "candidate_headsign": candidate.get("trip_headsign"),
        "candidate_direction_id": candidate.get("direction_id"),
        "candidate_shape_id": candidate.get("shape_id"),
    }

    return score, details


# ============================================================
# MATCH ONE UNIQUE EQASIM LINE/ROUTE PAIR
# ============================================================
def resolve_one_pair(
    transit_line_id,
    transit_route_id,
    direct_vehicle_lookup,
    line_to_meta,
    route_name_to_lines,
    gtfs_trip_lookup,
    stop_seq_lookup,
):
    """
    Matching strategy:

    CASE 1: direct match
    --------------------
    - look for vehicle id == transit_route_id + ".0"
    - if found:
        - check gtfs.route_name == X from transit_line_id
        - if yes => use this vehicle's "line"
        - if no => status = route_name_mismatch_direct

    CASE 2: fallback only if direct vehicle id is absent
    ---------------------------------------------------
    - find the commercial line of transit_route_id in GTFS
    - search in SUMO all vehicles/lines with gtfs.route_name == that commercial line
    - compare candidate SUMO lines against the original transit_route_id
      using GTFS metadata:
        - direction_id
        - headsign
        - shape_id
        - stop sequence
    - keep the exact/best match
    """

    expected_route_name = extract_line_suffix(transit_line_id)
    direct_vehicle_id = f"{transit_route_id}.0"

    # --------------------------------------------------------
    # CASE 1: DIRECT MATCH
    # --------------------------------------------------------
    # The direct case is the safest because it uses the expected vehicle ID.
    direct_match = direct_vehicle_lookup.get(direct_vehicle_id)

    if direct_match is not None:
        direct_route_name = direct_match.get("gtfs.route_name")

        if normalize_text(direct_route_name) == normalize_text(expected_route_name):
            return {
                "sumo_line": direct_match.get("line"),
                "match_method": "direct_vehicle_id",
                "match_status": "ok",
                "matched_vehicle_id": direct_vehicle_id,
                "expected_route_name": expected_route_name,
                "sumo_gtfs_route_name": direct_route_name,
                "sumo_gtfs_trip_headsign": direct_match.get("gtfs.trip_headsign"),
                "fallback_candidate_count": 0,
                "fallback_best_score": None,
                "fallback_reasons": None,
            }

        return {
            "sumo_line": None,
            "match_method": "direct_vehicle_id",
            "match_status": "route_name_mismatch_direct",
            "matched_vehicle_id": direct_vehicle_id,
            "expected_route_name": expected_route_name,
            "sumo_gtfs_route_name": direct_route_name,
            "sumo_gtfs_trip_headsign": direct_match.get("gtfs.trip_headsign"),
            "fallback_candidate_count": 0,
            "fallback_best_score": None,
            "fallback_reasons": None,
        }

    # --------------------------------------------------------
    # CASE 2: FALLBACK
    # --------------------------------------------------------
    target_trip = gtfs_trip_lookup.get(transit_route_id)
    if target_trip is None:
        return {
            "sumo_line": None,
            "match_method": "fallback",
            "match_status": "target_trip_not_in_gtfs",
            "matched_vehicle_id": None,
            "expected_route_name": expected_route_name,
            "sumo_gtfs_route_name": None,
            "sumo_gtfs_trip_headsign": None,
            "fallback_candidate_count": 0,
            "fallback_best_score": None,
            "fallback_reasons": None,
        }

    gtfs_commercial_route_name = target_trip.get("route_short_name")

    # Start from the commercial route name stored in the GTFS archive.
    commercial_route_name = gtfs_commercial_route_name

    # If the two sources disagree, prefer the Eqasim line identifier because
    # it describes the trip that is currently being mapped.
    if normalize_text(commercial_route_name) != normalize_text(expected_route_name):
        commercial_route_name = expected_route_name

    # Limit the fallback search to SUMO lines belonging to the same commercial route.
    candidate_lines = route_name_to_lines.get(str(commercial_route_name), [])

    if not candidate_lines:
        return {
            "sumo_line": None,
            "match_method": "fallback",
            "match_status": "no_candidate_line_for_route_name",
            "matched_vehicle_id": None,
            "expected_route_name": expected_route_name,
            "sumo_gtfs_route_name": None,
            "sumo_gtfs_trip_headsign": None,
            "fallback_candidate_count": 0,
            "fallback_best_score": None,
            "fallback_reasons": None,
        }

    scored_candidates = []
    for candidate_line in candidate_lines:
        score, details = score_candidate(
            target_trip_id=transit_route_id,
            candidate_line_trip_id=candidate_line,
            gtfs_trip_lookup=gtfs_trip_lookup,
            stop_seq_lookup=stop_seq_lookup,
        )
        scored_candidates.append((candidate_line, score, details))

    # Highest-scoring candidates are examined first.
    scored_candidates = sorted(scored_candidates, key=lambda x: x[1], reverse=True)

    best_line, best_score, best_details = scored_candidates[0]

    # Ambiguity check
    if len(scored_candidates) > 1 and scored_candidates[0][1] == scored_candidates[1][1]:
        return {
            "sumo_line": None,
            "match_method": "fallback",
            "match_status": "ambiguous_fallback",
            "matched_vehicle_id": None,
            "expected_route_name": expected_route_name,
            "sumo_gtfs_route_name": commercial_route_name,
            "sumo_gtfs_trip_headsign": None,
            "fallback_candidate_count": len(candidate_lines),
            "fallback_best_score": best_score,
            "fallback_reasons": " | ".join(best_details.get("reasons", [])),
        }

    # We accept fallback only if it is a strong match
    if not best_details.get("strong_match", False):
        return {
            "sumo_line": None,
            "match_method": "fallback",
            "match_status": "fallback_not_strong_enough",
            "matched_vehicle_id": None,
            "expected_route_name": expected_route_name,
            "sumo_gtfs_route_name": commercial_route_name,
            "sumo_gtfs_trip_headsign": None,
            "fallback_candidate_count": len(candidate_lines),
            "fallback_best_score": best_score,
            "fallback_reasons": " | ".join(best_details.get("reasons", [])),
        }

    sample_vehicle_id = safe_get(line_to_meta.get(best_line), "sample_vehicle_id")
    best_trip_headsign = safe_get(line_to_meta.get(best_line), "gtfs.trip_headsign")

    return {
        "sumo_line": best_line,
        "match_method": "fallback",
        "match_status": "ok",
        "matched_vehicle_id": sample_vehicle_id,
        "expected_route_name": expected_route_name,
        "sumo_gtfs_route_name": commercial_route_name,
        "sumo_gtfs_trip_headsign": best_trip_headsign,
        "fallback_candidate_count": len(candidate_lines),
        "fallback_best_score": best_score,
        "fallback_reasons": " | ".join(best_details.get("reasons", [])),
    }


# ============================================================
# BUILD THE COMPLETE MAPPING TABLE
# ============================================================
def build_mapping(
    eqasim_df,
    direct_vehicle_lookup,
    line_to_meta,
    route_name_to_lines,
    gtfs_trip_lookup,
    stop_seq_lookup,
):
    """Resolve each distinct Eqasim line/route pair only once."""

    # Reusing unique pairs avoids repeating the same expensive comparison.
    unique_pairs = (
        eqasim_df[["transit_line_id", "transit_route_id"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    records = []
    for _, row in unique_pairs.iterrows():
        result = resolve_one_pair(
            transit_line_id=row["transit_line_id"],
            transit_route_id=row["transit_route_id"],
            direct_vehicle_lookup=direct_vehicle_lookup,
            line_to_meta=line_to_meta,
            route_name_to_lines=route_name_to_lines,
            gtfs_trip_lookup=gtfs_trip_lookup,
            stop_seq_lookup=stop_seq_lookup,
        )

        record = {
            "transit_line_id": row["transit_line_id"],
            "transit_route_id": row["transit_route_id"],
            **result,
        }
        records.append(record)

    mapping_df = pd.DataFrame(records)
    return mapping_df


# ============================================================
# MAIN WORKFLOW
# ============================================================
def main():
    print("=== Loading input files ===")
    eqasim_df = load_eqasim_csv(EQASIM_CSV_PATH)
    direct_vehicle_lookup, line_to_meta, route_name_to_lines = load_sumo_pt_xml(
        SUMO_PT_XML_PATH
    )
    gtfs_trip_lookup, stop_seq_lookup = load_gtfs(GTFS_ZIP_PATH)

    print(f"Eqasim rows               : {len(eqasim_df)}")
    print(f"SUMO vehicles parsed      : {len(direct_vehicle_lookup)}")
    print(f"SUMO unique 'line' values : {len(line_to_meta)}")
    print(f"GTFS trips loaded         : {len(gtfs_trip_lookup)}")

    print("\n=== Building the unique line mapping ===")
    mapping_df = build_mapping(
        eqasim_df=eqasim_df,
        direct_vehicle_lookup=direct_vehicle_lookup,
        line_to_meta=line_to_meta,
        route_name_to_lines=route_name_to_lines,
        gtfs_trip_lookup=gtfs_trip_lookup,
        stop_seq_lookup=stop_seq_lookup,
    )

    print(f"Unique (transit_line_id, transit_route_id): {len(mapping_df)}")

    print("\n=== Merging the mapping with Eqasim trips ===")
    keep_columns = [
        "person_id",
        "person_trip_id",
        "leg_index",
        "transit_line_id",
        "transit_route_id",
    ]

    final_df = eqasim_df[keep_columns].merge(
        mapping_df,
        on=["transit_line_id", "transit_route_id"],
        how="left",
        validate="many_to_one",
    )

    # Compact output used by the downstream conversion pipeline.
    minimal_df = final_df[
        [
            "person_id",
            "person_trip_id",
            "leg_index",
            "transit_line_id",
            "transit_route_id",
            "sumo_line",
        ]
    ].copy()

    # Detailed output used to inspect and reproduce difficult matches.
    debug_df = final_df[
        [
            "person_id",
            "person_trip_id",
            "leg_index",
            "transit_line_id",
            "transit_route_id",
            "sumo_line",
            "match_method",
            "match_status",
            "matched_vehicle_id",
            "expected_route_name",
            "sumo_gtfs_route_name",
            "sumo_gtfs_trip_headsign",
            "fallback_candidate_count",
            "fallback_best_score",
            "fallback_reasons",
        ]
    ].copy()

    # Keep failed matches separate so they can be reviewed manually.
    unmatched_df = debug_df[debug_df["sumo_line"].isna()].copy()

    print("\n=== Export ===")
    minimal_df.to_csv(OUTPUT_MINIMAL_CSV, sep=";", index=False)
    debug_df.to_csv(OUTPUT_DEBUG_CSV, sep=";", index=False)
    unmatched_df.to_csv(OUTPUT_UNMATCHED_CSV, sep=";", index=False)

    print(f"Minimal mapping file : {OUTPUT_MINIMAL_CSV}")
    print(f"Diagnostic file      : {OUTPUT_DEBUG_CSV}")
    print(f"Unmatched records    : {OUTPUT_UNMATCHED_CSV}")

    print("\n=== Summary ===")
    print("Match status counts:")
    print(debug_df["match_status"].value_counts(dropna=False).to_string())

    matched_count = debug_df["sumo_line"].notna().sum()
    unmatched_count = debug_df["sumo_line"].isna().sum()

    print(f"\nRows matched     : {matched_count}")
    print(f"Rows unmatched   : {unmatched_count}")

    print("\nCompleted successfully.")


if __name__ == "__main__":
    main()

=== Loading input files ===
Eqasim rows               : 7706
SUMO vehicles parsed      : 1638
SUMO unique 'line' values : 240
GTFS trips loaded         : 3956

=== Building the unique line mapping ===
Unique (transit_line_id, transit_route_id): 838

=== Merging the mapping with Eqasim trips ===

=== Export ===
Minimal mapping file : output_tools/eqasim_pt_sumo_line_mapping.csv
Diagnostic file      : output_tools/eqasim_pt_sumo_line_mapping_debug.csv
Unmatched records    : output_tools/eqasim_pt_sumo_line_mapping_unmatched.csv

=== Summary ===
Match status counts:
match_status
ok                                  7637
target_trip_not_in_gtfs               37
no_candidate_line_for_route_name      32

Rows matched     : 7637
Rows unmatched   : 69

Completed successfully.


In [ ]:
# Copyright (c) 2026 Alix NGARI LENDOYE
# EIGSI La Rochelle
# La Rochelle Université — L3i
# SPDX-License-Identifier: GPL-3.0-only
import zstandard as zstd
import csv
import math
import io
import os
import re
import sys
import zipfile
import tempfile
import pickle
import atexit
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict, deque

import pandas as pd

# ============================================================
# Convert filtered MATSim population to SUMO persons
# PT strategy:
#   - use selected="yes" plan
#   - preserve original Eqasim/MATSim trip indices after merging activities
#   - take SUMO PT lines directly from eqasim_pt_sumo_line_mapping.csv
#   - prefer exact Quay names from access_stop_id / egress_stop_id
#   - fallback to StopPlace names from access_area_id / egress_area_id
#   - if needed, allow generic -> specific matching:
#       "Dames Blanches" -> "Dames Blanches (St-Nicolas)"
#   - if a PT trip is explicitly listed in the unmatched file,
#     fallback that trip to car_passenger (taxi hypothesis)
#   - PT stages are now written in busStop form:
#       walk edge->busStop, ride busStop->busStop, walk busStop->edge
# ============================================================

# ============================================================
# PATHS (edit if needed)
# ============================================================
POP_PATH      = Path(r"../eqasim_output_filtered/output_plans_filtered.xml.zst")
CSV_PATH      = Path(r"../2-POI's/facilities2sumo_multimode.csv")
STOPS_ADD     = Path(r"../3-public_transport/gtfs_pt_stops.add.xml")
VEHS_ADD      = Path(r"../3-public_transport/gtfs_pt_vehicles.add.xml")
NET_PATH      = Path(r"../1-network/cda_la_rochelle.net.xml")   # accepts .net.xml or .zip
GTFS_ZIP      = Path(r"../3-public_transport/input/ca_la_rochelle-aggregated-gtfs.zip")

EQASIM_PT_CSV = Path(r"../eqasim_output_filtered/eqasim_pt_filtered.csv")
PT_MAP_CSV    = Path(r"./output_tools/eqasim_pt_sumo_line_mapping.csv")
PT_UNMATCHED  = Path(r"./output_tools/eqasim_pt_sumo_line_mapping_unmatched.csv")

OUT_ROU       = Path("population_all.rou.xml")
OUT_LOG       = Path("output_tools/population_all.log.csv")

# ============================================================
# OPTIONS
# ============================================================
FORCE_LAST_STOP_TO_86400 = True
PRINT_EVERY = 100

# ============================================================
# NEW NETWORK SAFETY OPTIONS
# ============================================================
# Important for networks generated with --sidewalks.guess, --crossings.guess
# and --walkingareas true. Without this, sumolib may report false noRoute
# for pedestrian paths that pass through crossings / walking areas.
READ_NET_WITH_PEDESTRIAN_CONNECTIONS = True

# Optional filters for quick tests.
# Example: TEST_IDS = {"122556", "122590"}
TEST_IDS = None
TEST_MAX_PERSONS = None

# Keep the old conversion logic as the main logic. Candidate fallback is only
# used for pedestrian walk segments when the primary pedestrian anchors fail.
USE_PED_CANDIDATE_FALLBACK = True
PED_CANDIDATES_CSV = Path(r"../2-POI's/facilities2sumo_pedestrian_candidates.csv")
PED_CANDIDATE_LIMIT = 2

# Do not write personTrip modes="car" from an edge that does not allow passenger.
# Instead, skip the person with a clear log reason.
SKIP_UNSAFE_VEHICLE_START_EDGE = True

ENABLE_CONNECTIVITY_PRECHECK = False  # V3: writing phase checks only the necessary transitions
ENABLE_COMPONENT_DIAG = False       # V3: avoid expensive BFS diagnostics during production runs
COMPONENT_BFS_MAX_VISITS = 250_000

# if SUMO shortest path cost is length in meters, keep False
WALK_SPEED = 1.3
COST_IS_TIME = False

# ============================================================
# SUMO / sumolib
# ============================================================
SUMO_HOME = os.environ.get("SUMO_HOME")
if not SUMO_HOME:
    raise RuntimeError(
        "SUMO_HOME is not set. Please set SUMO_HOME to your SUMO installation folder "
        "(it must contain the 'tools' directory)."
    )

tools_path = str(Path(SUMO_HOME) / "tools")
if tools_path not in sys.path:
    sys.path.append(tools_path)

import sumolib


def _extract_netxml_from_zip(zip_path: Path) -> Path:
    with zipfile.ZipFile(zip_path, "r") as zf:
        net_members = [n for n in zf.namelist() if n.endswith(".net.xml")]
        if not net_members:
            raise ValueError(f"No .net.xml found in {zip_path}")
        member = net_members[0]
        tmpdir = Path(tempfile.mkdtemp(prefix="sumo_net_"))
        out = tmpdir / Path(member).name
        with zf.open(member) as src, out.open("wb") as dst:
            dst.write(src.read())
        return out


def _read_sumo_net(net_path: Path):
    kwargs = {}
    if READ_NET_WITH_PEDESTRIAN_CONNECTIONS:
        kwargs.update({
            "withPedestrianConnections": True,
            "withFoes": False,
            "withPrograms": False,
        })
    return sumolib.net.readNet(str(net_path), **kwargs)


def load_net(path: Path):
    print("[INFO] Loading SUMO network...")
    print(f"[INFO] Network file: {path}")
    print(f"[INFO] withPedestrianConnections={READ_NET_WITH_PEDESTRIAN_CONNECTIONS}")
    if path.suffix.lower() == ".zip":
        netxml = _extract_netxml_from_zip(path)
        return _read_sumo_net(netxml)
    return _read_sumo_net(path)


net = load_net(NET_PATH)

# ============================================================
# Helpers
# ============================================================
def ln(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag


def hms_to_seconds(hms: str) -> int:
    h, m, s = hms.strip().split(":")
    return int(h) * 3600 + int(m) * 60 + int(s)


def fmt_pos(x: float) -> str:
    return f"{float(x):.2f}".rstrip("0").rstrip(".")


def is_nan_value(x) -> bool:
    if x is None:
        return True
    if isinstance(x, float):
        return math.isnan(x)
    s = str(x).strip().lower()
    return s in {"nan", "none", ""}


def is_interaction_activity(act_elem: ET.Element) -> bool:
    t = act_elem.get("type") or ""
    return "interaction" in t.lower()


def lane_to_edge(lane_id: str) -> str:
    return lane_id.rsplit("_", 1)[0] if "_" in lane_id else lane_id


def normalize_name(x: str) -> str:
    if x is None:
        return ""
    s = str(x).strip().lower()
    s = s.replace("’", "'")
    s = re.sub(r"\s+", " ", s)
    return s


def generic_prefix_of_name(x: str) -> str:
    """
    'Dames Blanches (St-Nicolas)' -> 'dames blanches'
    'Jaurès' -> 'jaurès'
    """
    s = normalize_name(x)
    s = re.sub(r"\s*\(.*\)\s*$", "", s).strip()
    return s


def stop_name_matches(query_name: str, candidate_name: str) -> bool:
    """
    Exact first.
    Then allow generic -> specific:
      'Dames Blanches' matches 'Dames Blanches (St-Nicolas)'
    But not the reverse.
    """
    q = normalize_name(query_name)
    c = normalize_name(candidate_name)
    if not q or not c:
        return False
    if c == q:
        return True
    if "(" not in q and c.startswith(q + " ("):
        return True
    return False


# --- edge object cache ---
_edge_obj_cache = {}


def edge_obj(edge_id: str):
    if edge_id in _edge_obj_cache:
        return _edge_obj_cache[edge_id]
    e = net.getEdge(edge_id)
    if e is None:
        raise ValueError(f"unknown edge in net: {edge_id}")
    _edge_obj_cache[edge_id] = e
    return e


# --- shortest path cost cache (walk/car/bike only) ---
# V3: persistent cache. The network is already loaded once, but shortest-path
# calls are still expensive. This cache avoids recomputing the same edge pairs
# across repeated test/full runs. Delete this file if you change the network.
ROUTE_CACHE_PATH = Path("shortest_path_cache_s0_new_network_v3.pkl")
ROUTE_CACHE_SAVE_EVERY = 5000
_sp_cost_cache = {}
_route_cache_new_entries = 0
_route_cache_hits = 0
_route_cache_misses = 0

if ROUTE_CACHE_PATH.exists():
    try:
        with ROUTE_CACHE_PATH.open("rb") as f:
            obj = pickle.load(f)
        if isinstance(obj, dict):
            _sp_cost_cache.update(obj)
        print(f"[INFO] Loaded shortest-path cache: {len(_sp_cost_cache)} entries from {ROUTE_CACHE_PATH}")
    except Exception as e:
        print(f"[WARN] Could not load shortest-path cache {ROUTE_CACHE_PATH}: {e}")


def save_route_cache():
    try:
        with ROUTE_CACHE_PATH.open("wb") as f:
            pickle.dump(_sp_cost_cache, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"[INFO] Saved shortest-path cache: {len(_sp_cost_cache)} entries to {ROUTE_CACHE_PATH}")
    except Exception as e:
        print(f"[WARN] Could not save shortest-path cache {ROUTE_CACHE_PATH}: {e}")


atexit.register(save_route_cache)


def shortest_cost(from_edge: str, to_edge: str, vclass: str) -> float:
    global _route_cache_new_entries, _route_cache_hits, _route_cache_misses

    # Same edge: no full graph search needed.
    if from_edge == to_edge:
        return 0.0

    key = (from_edge, to_edge, vclass)
    if key in _sp_cost_cache:
        _route_cache_hits += 1
        return _sp_cost_cache[key]

    _route_cache_misses += 1
    fe = edge_obj(from_edge)
    te = edge_obj(to_edge)
    path, cost = net.getShortestPath(fe, te, vClass=vclass)
    if path is None:
        val = float("inf")
    else:
        val = float(cost) if cost is not None else 0.0

    _sp_cost_cache[key] = val
    _route_cache_new_entries += 1

    if _route_cache_new_entries % ROUTE_CACHE_SAVE_EVERY == 0:
        save_route_cache()

    return val


def has_route(from_edge: str, to_edge: str, vclass: str) -> bool:
    return math.isfinite(shortest_cost(from_edge, to_edge, vclass))


def edge_allows_vclass(edge_id: str, vclass: str) -> bool:
    """True if at least one lane of edge_id allows vclass."""
    try:
        e = edge_obj(edge_id)
    except Exception:
        return False
    for ln_ in e.getLanes():
        if ln_.allows(vclass):
            return True
    return False




def first_lane_for_vclass(edge_id: str, vclass: str = None) -> str:
    """Return a lane id on edge_id, preferring a lane that allows vclass."""
    e = edge_obj(edge_id)
    lanes = e.getLanes()
    if not lanes:
        raise ValueError(f"edge has no lanes: {edge_id}")
    if vclass is not None:
        for ln_ in lanes:
            if ln_.allows(vclass):
                return ln_.getID()
    return lanes[0].getID()

def require_edge_allows(edge_id: str, vclass: str, context: str = ""):
    if not edge_allows_vclass(edge_id, vclass):
        suffix = f" context={context}" if context else ""
        raise ValueError(f"edge not allowed for vclass={vclass}: {edge_id}{suffix}")


def connect_activity_to_vehicle_start(buf, from_edge, from_pos, target_edge, target_pos, target_vclass, context=""):
    """
    Preserve the old logic: if a pedestrian access path exists from the current
    activity stop to the vehicle anchor, write it.

    New-network safety: if no such pedestrian access exists, never replace the
    vehicle start by a pedestrian-only edge. That was the source of SUMO warnings
    such as DEFAULT_VEHTYPE not allowed on the start edge.
    
    If the current edge itself already allows the target vehicle class, we can
    safely start there. Otherwise, skip this person with a clear error.
    """
    if from_edge == target_edge and abs(float(from_pos) - float(target_pos)) <= 1e-6:
        require_edge_allows(target_edge, target_vclass, context)
        return target_edge, target_pos

    if has_route(from_edge, target_edge, "pedestrian"):
        buf.write(
            f'        <walk from="{from_edge}" to="{target_edge}" '
            f'arrivalPos="{fmt_pos(target_pos)}" />\n'
        )
        require_edge_allows(target_edge, target_vclass, context)
        return target_edge, target_pos

    if edge_allows_vclass(from_edge, target_vclass):
        return from_edge, from_pos

    if SKIP_UNSAFE_VEHICLE_START_EDGE:
        reason = connectivity_reason(from_edge, target_edge, "pedestrian")
        raise ValueError(
            f"no pedestrian access to {target_vclass} start edge {from_edge}->{target_edge} "
            f"context={context} ({reason})"
        )

    # Last-resort legacy behavior, disabled by default for new networks.
    return from_edge, from_pos


# ============================================================
# Connectivity diagnostic
# ============================================================
_component_reach_cache = {}


def _edge_allows_vclass(edge, vclass: str):
    for ln_ in edge.getLanes():
        if ln_.allows(vclass):
            return True
    return False


def _neighbors_for_vclass(edge, vclass: str):
    for e2 in edge.getOutgoing().keys():
        if _edge_allows_vclass(e2, vclass):
            yield e2


def same_component_bfs(from_edge_id: str, to_edge_id: str, vclass: str):
    key = (from_edge_id, vclass)
    if key in _component_reach_cache:
        visited_set, visited_count, _ = _component_reach_cache[key]
        if visited_set is not None:
            return (to_edge_id in visited_set), visited_count

    start = edge_obj(from_edge_id)
    goal = to_edge_id

    q = deque([start])
    visited = set([from_edge_id])
    visits = 0

    while q:
        e = q.popleft()
        visits += 1
        if e.getID() == goal:
            _component_reach_cache[key] = (visited, len(visited), True)
            return True, len(visited)
        if visits > COMPONENT_BFS_MAX_VISITS:
            _component_reach_cache[key] = (None, visits, False)
            return False, visits
        for n in _neighbors_for_vclass(e, vclass):
            nid = n.getID()
            if nid not in visited:
                visited.add(nid)
                q.append(n)

    _component_reach_cache[key] = (visited, len(visited), True)
    return False, len(visited)


def connectivity_reason(from_edge: str, to_edge: str, vclass: str) -> str:
    if has_route(from_edge, to_edge, vclass):
        return ""
    if not ENABLE_COMPONENT_DIAG:
        return "no_route"
    ok, sz = same_component_bfs(from_edge, to_edge, vclass)
    if ok:
        return "no_route"
    return f"disconnected components (vclass={vclass}, component_visits~{sz})"


# ============================================================
# Robust CSV parsing
# ============================================================
def read_csv_auto(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"CSV not found: {path.resolve()}")
    header = path.read_text(encoding="utf-8", errors="replace").splitlines()[0]
    if ";" in header and "," not in header:
        sep = ";"
    elif "," in header and ";" not in header:
        sep = ","
    else:
        sep = None
    df_ = pd.read_csv(path, sep=sep, engine="python", encoding="utf-8-sig")
    df_.columns = [str(c).strip().lstrip("\ufeff") for c in df_.columns]
    return df_


# ============================================================
# facilities2sumo mapping
# ============================================================
df = read_csv_auto(CSV_PATH)

if "status" in df.columns:
    df = df[df["status"] == "mapped"].copy()

required = {"poi_id", "mode", "edge_id", "lane_id", "pos"}
missing = required - set(df.columns)
if missing:
    raise ValueError(
        f"facilities2sumo_multimode.csv is missing columns: {missing}. "
        f"Available columns: {list(df.columns)}"
    )

fac_map = {(str(r.poi_id), str(r.mode)): (r.edge_id, r.lane_id, r.pos) for r in df.itertuples(index=False)}

# Optional pedestrian candidate fallback. This does not replace the main
# facilities2sumo_multimode.csv mapping; it is only used when a primary walk
# segment has no pedestrian route.
ped_candidates_by_poi = defaultdict(list)
if USE_PED_CANDIDATE_FALLBACK and PED_CANDIDATES_CSV.exists():
    cand_df = read_csv_auto(PED_CANDIDATES_CSV)
    cand_df = cand_df[cand_df.get("status", "mapped") == "mapped"].copy() if "status" in cand_df.columns else cand_df.copy()
    required_cand = {"poi_id", "edge_id", "lane_id", "pos"}
    missing_cand = required_cand - set(cand_df.columns)
    if missing_cand:
        raise ValueError(f"{PED_CANDIDATES_CSV} missing columns: {missing_cand}")
    sort_cols = [c for c in ["poi_id", "candidate_rank", "dist_to_edge"] if c in cand_df.columns]
    if sort_cols:
        cand_df = cand_df.sort_values(sort_cols)
    for r in cand_df.itertuples(index=False):
        poi = str(getattr(r, "poi_id"))
        edge = str(getattr(r, "edge_id"))
        lane = str(getattr(r, "lane_id"))
        pos = float(getattr(r, "pos"))
        if not is_nan_value(edge) and not is_nan_value(lane) and not is_nan_value(pos):
            ped_candidates_by_poi[poi].append((edge, lane, pos))
    print(f"[INFO] Pedestrian candidate fallback loaded: {len(ped_candidates_by_poi)} POIs")
elif USE_PED_CANDIDATE_FALLBACK:
    print(f"[WARN] USE_PED_CANDIDATE_FALLBACK=True but file not found: {PED_CANDIDATES_CSV}")


def get_ped_candidates(poi_id: str):
    """Primary pedestrian mapping first, then optional candidates without duplicates."""
    out = []
    seen = set()
    try:
        edge, lane, pos = get_fac_sumo(poi_id, "pedestrian")
        key = (edge, lane, round(float(pos), 2))
        out.append((edge, lane, float(pos)))
        seen.add(key)
    except Exception:
        pass

    if USE_PED_CANDIDATE_FALLBACK:
        for edge, lane, pos in ped_candidates_by_poi.get(str(poi_id), [])[:PED_CANDIDATE_LIMIT]:
            key = (edge, lane, round(float(pos), 2))
            if key not in seen:
                out.append((edge, lane, float(pos)))
                seen.add(key)

    return out


def choose_pedestrian_target_from_current(current_edge, current_pos, to_fac):
    """
    V3 stateful pedestrian choice.

    The current stop is already written where the person really is. For a walk
    segment, we only need to choose a reachable pedestrian anchor for the next
    activity. We do NOT test every departure-candidate × arrival-candidate pair.

    Returns dst_edge, dst_lane, dst_pos.
    """
    candidates = get_ped_candidates(to_fac)
    if not candidates:
        raise ValueError(f"no pedestrian candidates for facility={to_fac}")

    first_edge, _first_lane, _first_pos = candidates[0]

    for dst_edge, dst_lane, dst_pos in candidates:
        edge_obj(dst_edge)

        # Same edge: no graph search needed.
        if current_edge == dst_edge:
            return dst_edge, dst_lane, dst_pos

        if has_route(current_edge, dst_edge, "pedestrian"):
            return dst_edge, dst_lane, dst_pos

    reason = connectivity_reason(current_edge, first_edge, "pedestrian")
    raise ValueError(f"no pedestrian route from current edge to candidates {current_edge}->{first_edge} ({reason})")


# Legacy function name kept for compatibility, but V3 writer does not use the
# combinatorial departure×arrival search anymore.
def choose_pedestrian_walk_from_current(current_edge, current_pos, from_fac, to_fac):
    dst_edge, dst_lane, dst_pos = choose_pedestrian_target_from_current(current_edge, current_pos, to_fac)
    return current_edge, current_pos, dst_edge, dst_pos


def get_fac_sumo(poi_id: str, preferred_mode: str):
    for m in (preferred_mode, "passenger", "pedestrian"):
        k = (poi_id, m)
        if k in fac_map:
            edge, lane, pos = fac_map[k]
            if is_nan_value(edge) or is_nan_value(lane) or is_nan_value(pos):
                raise ValueError(f"nan mapping for facility={poi_id} mode={m} -> edge={edge}, lane={lane}, pos={pos}")
            return str(edge), str(lane), float(pos)

    for (p, _m), (edge, lane, pos) in fac_map.items():
        if p == poi_id:
            if is_nan_value(edge) or is_nan_value(lane) or is_nan_value(pos):
                raise ValueError(f"nan mapping for facility={poi_id} (fallback) -> edge={edge}, lane={lane}, pos={pos}")
            return str(edge), str(lane), float(pos)

    raise KeyError(f"Missing facility mapping for {poi_id!r}")


def get_fac_sumo_strict(poi_id: str, mode: str):
    k = (poi_id, mode)
    if k not in fac_map:
        raise KeyError(f"Missing strict facility mapping for {poi_id!r} mode={mode!r}")
    edge, lane, pos = fac_map[k]
    if is_nan_value(edge) or is_nan_value(lane) or is_nan_value(pos):
        raise ValueError(f"nan mapping for facility={poi_id} mode={mode} -> edge={edge}, lane={lane}, pos={pos}")
    return str(edge), str(lane), float(pos)


# ============================================================
# Eqasim PT + mapping CSVs
# ============================================================
eq_pt = read_csv_auto(EQASIM_PT_CSV)
pt_map = read_csv_auto(PT_MAP_CSV)
pt_unmatched = read_csv_auto(PT_UNMATCHED)

req_eq = {
    "person_id", "person_trip_id", "leg_index",
    "access_stop_id", "egress_stop_id",
    "access_area_id", "egress_area_id",
    "transit_line_id", "transit_route_id"
}
req_map = {
    "person_id", "person_trip_id", "leg_index",
    "transit_line_id", "transit_route_id", "sumo_line"
}
req_unmatched = {
    "person_id", "person_trip_id", "leg_index",
    "transit_line_id", "transit_route_id"
}

missing_eq = req_eq - set(eq_pt.columns)
missing_map = req_map - set(pt_map.columns)
missing_unmatched = req_unmatched - set(pt_unmatched.columns)
if missing_eq:
    raise ValueError(f"eqasim_pt_filtered.csv missing columns: {missing_eq}")
if missing_map:
    raise ValueError(f"eqasim_pt_sumo_line_mapping.csv missing columns: {missing_map}")
if missing_unmatched:
    raise ValueError(f"eqasim_pt_sumo_line_mapping_unmatched.csv missing columns: {missing_unmatched}")

pt_join = eq_pt.merge(
    pt_map,
    on=["person_id", "person_trip_id", "leg_index", "transit_line_id", "transit_route_id"],
    how="left",
    validate="one_to_one"
)

pt_join = pt_join[~pt_join["sumo_line"].isna()].copy()

pt_rows_by_trip = defaultdict(list)
for r in pt_join.sort_values(["person_id", "person_trip_id", "leg_index"]).itertuples(index=False):
    pt_rows_by_trip[(str(r.person_id), int(r.person_trip_id))].append(r)

unmatched_pairs = set()
for r in pt_unmatched.itertuples(index=False):
    unmatched_pairs.add((str(r.person_id), int(r.person_trip_id)))


# ============================================================
# GTFS stop names
# ============================================================
if not GTFS_ZIP.exists():
    raise FileNotFoundError(f"GTFS zip not found: {GTFS_ZIP.resolve()}")

with zipfile.ZipFile(GTFS_ZIP) as zf:
    with zf.open("stops.txt") as f:
        gtfs_stops = pd.read_csv(f, low_memory=False)

gtfs_stops.columns = [str(c).strip() for c in gtfs_stops.columns]

stop_id_to_name = {}
parent_station_of = {}

for r in gtfs_stops.itertuples(index=False):
    sid = str(getattr(r, "stop_id"))
    name = str(getattr(r, "stop_name")) if not pd.isna(getattr(r, "stop_name")) else ""
    stop_id_to_name[sid] = name
    parent = getattr(r, "parent_station", None)
    if pd.notna(parent):
        parent_station_of[sid] = str(parent)


def strip_quay_link(stop_id: str) -> str:
    return re.sub(r"\.link:.*$", "", str(stop_id))


def resolve_name_from_stop_id(stop_id: str, prefer_parent: bool = False) -> str:
    """
    For Quay IDs:
      - prefer_parent=False -> exact Quay stop_name first
      - prefer_parent=True  -> parent StopPlace name if available
    """
    if not stop_id or is_nan_value(stop_id):
        return ""

    sid = str(stop_id).strip()
    candidates = [sid, strip_quay_link(sid)]

    # exact stop_id / stripped Quay
    for c in candidates:
        if c in stop_id_to_name:
            if prefer_parent:
                parent = parent_station_of.get(c)
                if parent and parent in stop_id_to_name:
                    return stop_id_to_name[parent]
            return stop_id_to_name[c]

    # parent fallback
    for c in candidates:
        parent = parent_station_of.get(c)
        if parent and parent in stop_id_to_name:
            return stop_id_to_name[parent]

    return ""


def unique_nonempty(items):
    out = []
    seen = set()
    for x in items:
        s = str(x).strip() if x is not None else ""
        if not s:
            continue
        n = normalize_name(s)
        if n not in seen:
            seen.add(n)
            out.append(s)
    return out


# ============================================================
# Parse SUMO PT routes from vehicles.add.xml
# ============================================================
if not VEHS_ADD.exists():
    raise FileNotFoundError(f"PT vehicles file not found: {VEHS_ADD.resolve()}")

veh_root = ET.parse(VEHS_ADD).getroot()

route_edges = {}
route_stop_ids = {}
available_lines = set()

for ch in veh_root:
    tag = ln(ch.tag)
    if tag == "route":
        rid = ch.get("id")
        edges = (ch.get("edges") or "").split()
        stops = []
        for sub in list(ch):
            if ln(sub.tag) != "stop":
                continue
            bus_stop_id = sub.get("busStop")
            if bus_stop_id:
                stops.append(bus_stop_id)
        if rid and edges:
            route_edges[rid] = edges
            route_stop_ids[rid] = stops
    elif tag == "vehicle":
        line = ch.get("line")
        if line:
            available_lines.add(line)


# ============================================================
# Parse SUMO PT stops from gtfs_pt_stops.add.xml
# ============================================================
if not STOPS_ADD.exists():
    raise FileNotFoundError(f"PT stops file not found: {STOPS_ADD.resolve()}")

all_bus_stops = []
bus_stop_by_id = {}

st_root = ET.parse(STOPS_ADD).getroot()
for bs in st_root.iter():
    if ln(bs.tag) != "busStop":
        continue

    bs_id = bs.get("id")
    lane = bs.get("lane")
    name = bs.get("name") or ""
    s = bs.get("startPos")
    e = bs.get("endPos")

    if not bs_id or not lane or s is None:
        continue

    try:
        s = float(s)
        pos = (s + float(e)) / 2.0 if e is not None else s
    except Exception:
        continue

    edge = lane_to_edge(lane)
    info = {
        "busStopId": bs_id,
        "name": name,
        "name_norm": normalize_name(name),
        "name_generic": generic_prefix_of_name(name),
        "lane": lane,
        "edge": edge,
        "pos": float(pos),
    }
    all_bus_stops.append(info)
    bus_stop_by_id[bs_id] = info

_line_stops_cache = {}


def get_line_stops(line: str):
    if line in _line_stops_cache:
        return _line_stops_cache[line]

    # Preferred path: exact route stop sequence from gtfs_pt_vehicles.add.xml
    stop_ids = route_stop_ids.get(line, [])
    if stop_ids:
        ordered = []
        for bs_id in stop_ids:
            st = bus_stop_by_id.get(bs_id)
            if st is None:
                continue
            ordered.append(dict(st))
        if ordered:
            _line_stops_cache[line] = ordered
            return ordered

    # Fallback: recover stops by route edges if explicit stop list is unavailable
    if line not in route_edges:
        raise ValueError(f"SUMO line not found as route id in vehicles.add.xml: {line}")

    edges = route_edges[line]
    edge_first_idx = {}
    for i, e in enumerate(edges):
        edge_first_idx.setdefault(e, i)

    cand = [st for st in all_bus_stops if st["edge"] in edge_first_idx]
    if not cand:
        raise ValueError(f"no busStop found on SUMO route edges for line: {line}")

    cand.sort(key=lambda st: (edge_first_idx[st["edge"]], st["pos"], st["busStopId"]))
    out = [dict(st) for st in cand]
    _line_stops_cache[line] = out
    return out


def _find_named_stop_after_single(line: str, stop_name: str, min_idx: int = 0):
    lst = get_line_stops(line)
    for idx in range(max(0, min_idx), len(lst)):
        if stop_name_matches(stop_name, lst[idx]["name"]):
            return idx, lst[idx]
    return None, None


def find_named_stop_after(line: str, primary_name: str, min_idx: int = 0, fallback_names=None):
    """
    Tries in order:
      1) primary_name
      2) each fallback name
    Matching is exact first, with generic->specific allowed.
    """
    names = unique_nonempty([primary_name] + list(fallback_names or []))
    for nm in names:
        idx, st = _find_named_stop_after_single(line, nm, min_idx)
        if st is not None:
            return idx, st, nm
    return None, None, None


def first_common_transfer(line1: str, min_idx1: int, line2: str):
    lst1 = get_line_stops(line1)
    lst2 = get_line_stops(line2)

    # exact common names first
    names2_exact = defaultdict(list)
    for j, st in enumerate(lst2):
        names2_exact[st["name_norm"]].append((j, st))

    for i in range(max(0, min_idx1), len(lst1)):
        st1 = lst1[i]
        nm = st1["name_norm"]
        if nm and nm in names2_exact:
            j, st2 = names2_exact[nm][0]
            return (i, st1), (j, st2)

    # generic common names fallback
    names2_generic = defaultdict(list)
    for j, st in enumerate(lst2):
        names2_generic[st["name_generic"]].append((j, st))

    for i in range(max(0, min_idx1), len(lst1)):
        st1 = lst1[i]
        nm = st1["name_generic"]
        if nm and nm in names2_generic:
            j, st2 = names2_generic[nm][0]
            return (i, st1), (j, st2)

    return (None, None), (None, None)


def find_stop_index_by_id(line: str, bus_stop_id: str):
    lst = get_line_stops(line)
    for idx, st in enumerate(lst):
        if st["busStopId"] == bus_stop_id:
            return idx, st
    return None, None


# ============================================================
# Build PT chain from Eqasim PT rows
# ============================================================
def build_pt_chain_for_trip(person_id: str, person_trip_id: int):
    rows = pt_rows_by_trip.get((str(person_id), int(person_trip_id)), [])
    if not rows:
        raise ValueError(f"no PT mapping rows for person_id={person_id}, person_trip_id={person_trip_id}")

    legs = []
    for r in rows:
        sumo_line = str(r.sumo_line)

        if sumo_line not in available_lines and sumo_line not in route_edges:
            raise ValueError(f"mapped SUMO line not present in vehicles.add.xml: {sumo_line}")

        # primary = exact Quay stop_name
        access_name_primary = resolve_name_from_stop_id(getattr(r, "access_stop_id", None), prefer_parent=False)
        egress_name_primary = resolve_name_from_stop_id(getattr(r, "egress_stop_id", None), prefer_parent=False)

        # fallback = StopPlace name
        access_name_fallback = resolve_name_from_stop_id(getattr(r, "access_area_id", None), prefer_parent=False)
        egress_name_fallback = resolve_name_from_stop_id(getattr(r, "egress_area_id", None), prefer_parent=False)

        legs.append({
            "leg_index": int(r.leg_index),
            "sumo_line": sumo_line,
            "access_name_primary": access_name_primary,
            "access_name_fallbacks": unique_nonempty([access_name_fallback]),
            "egress_name_primary": egress_name_primary,
            "egress_name_fallbacks": unique_nonempty([egress_name_fallback]),
            "access_stop_id": str(getattr(r, "access_stop_id", "")),
            "egress_stop_id": str(getattr(r, "egress_stop_id", "")),
            "access_area_id": str(getattr(r, "access_area_id", "")),
            "egress_area_id": str(getattr(r, "egress_area_id", "")),
        })

    legs.sort(key=lambda x: x["leg_index"])
    return legs


# ============================================================
# MATSim parsing
# ============================================================
def get_selected_plan(person_elem: ET.Element):
    plans = [ch for ch in list(person_elem) if ln(ch.tag) == "plan"]
    if not plans:
        return None
    for p in plans:
        if p.get("selected") == "yes":
            return p
    return plans[0]


def get_child_attr_value(elem: ET.Element, attr_name: str):
    for ch in elem:
        if ln(ch.tag) == "attributes":
            for a in ch:
                if ln(a.tag) == "attribute" and a.get("name") == attr_name:
                    return (a.text or "").strip()
    return None


def build_activity_leg_sequence(plan: ET.Element):
    children = list(plan)
    kept_idx = [i for i, ch in enumerate(children)
                if ln(ch.tag) == "activity" and not is_interaction_activity(ch)]
    acts = [children[i] for i in kept_idx]
    if not acts:
        return [], []

    original_segments = []
    for trip_idx, idx in enumerate(kept_idx[:-1]):
        nxt = kept_idx[trip_idx + 1]
        legs = [ch for ch in children[idx + 1:nxt] if ln(ch.tag) == "leg"]
        original_segments.append({
            "orig_trip_id": trip_idx,
            "legs": legs,
        })

    return acts, original_segments


def merge_same_facility_preserve_trip_ids(acts, original_segments):
    merged_acts = []
    group_starts = []

    for i, act in enumerate(acts):
        if not merged_acts:
            merged_acts.append(act)
            group_starts.append(i)
            continue

        if act.get("facility") == merged_acts[-1].get("facility"):
            if act.get("end_time") is not None:
                merged_acts[-1].set("end_time", act.get("end_time"))
            if (merged_acts[-1].get("type") in (None, "", "unknown")) and act.get("type"):
                merged_acts[-1].set("type", act.get("type"))
        else:
            merged_acts.append(act)
            group_starts.append(i)

    merged_segments = []
    for g in range(len(merged_acts) - 1):
        start_idx = group_starts[g]
        end_idx = group_starts[g + 1]

        agg_legs = []
        leg_groups = []
        orig_trip_ids = []

        for k in range(start_idx, end_idx):
            seg = original_segments[k]
            agg_legs.extend(seg["legs"])
            leg_groups.append({
                "orig_trip_id": seg["orig_trip_id"],
                "legs": seg["legs"],
            })
            orig_trip_ids.append(seg["orig_trip_id"])

        merged_segments.append({
            "legs": agg_legs,
            "leg_groups": leg_groups,
            "orig_trip_ids": orig_trip_ids,
        })

    return merged_acts, merged_segments


def classify_car_role(legs):
    role = None
    for leg in legs:
        mode = leg.get("mode", "walk")
        routing = get_child_attr_value(leg, "routingMode")
        if mode in ("car", "car_passenger") or routing in ("car", "car_passenger"):
            this = "car_passenger" if (mode == "car_passenger" or routing == "car_passenger") else "car"
            if role != "car_passenger":
                role = this
    return role


def has_any_pt(legs):
    for leg in legs:
        mode = leg.get("mode", "walk")
        routing = get_child_attr_value(leg, "routingMode")
        if mode == "pt" or routing == "pt":
            return True
    return False


def has_any_bike(legs):
    for leg in legs:
        mode = (leg.get("mode") or "").lower()
        routing = (get_child_attr_value(leg, "routingMode") or "").lower()
        if mode in ("bike", "bicycle") or routing in ("bike", "bicycle"):
            return True
    return False


def resolve_original_pt_trip_id(person_id: str, merged_seg: dict) -> int:
    pt_trip_candidates = []
    for grp in merged_seg["leg_groups"]:
        if has_any_pt(grp["legs"]):
            pt_trip_candidates.append(int(grp["orig_trip_id"]))

    if not pt_trip_candidates:
        raise ValueError(f"merged PT segment has no original PT trip for person_id={person_id}")

    mapped_candidates = [t for t in pt_trip_candidates if (str(person_id), int(t)) in pt_rows_by_trip]

    if len(mapped_candidates) == 1:
        return mapped_candidates[0]

    if len(mapped_candidates) > 1:
        raise ValueError(
            f"multiple PT mapping rows candidates for person_id={person_id}, "
            f"candidate_trip_ids={mapped_candidates}"
        )

    unmatched_candidates = [t for t in pt_trip_candidates if (str(person_id), int(t)) in unmatched_pairs]

    if len(unmatched_candidates) == 1:
        return unmatched_candidates[0]

    if len(unmatched_candidates) > 1:
        raise ValueError(
            f"multiple unmatched PT trips candidates for person_id={person_id}, "
            f"candidate_trip_ids={unmatched_candidates}"
        )

    if len(pt_trip_candidates) == 1:
        return pt_trip_candidates[0]

    raise ValueError(
        f"multiple original PT trips merged but none mapped for person_id={person_id}, "
        f"candidate_trip_ids={pt_trip_candidates}"
    )


def effective_segment_kind_for_arrival(person_id: str, merged_seg: dict, nominal_kind: str) -> str:
    """
    Actual mode that reaches the next activity.
    Needed because a nominal PT segment can fallback to taxi (car_passenger).
    """
    if nominal_kind != "pt":
        return nominal_kind

    try:
        original_pt_trip_id = resolve_original_pt_trip_id(person_id, merged_seg)
    except Exception:
        return nominal_kind

    if (str(person_id), int(original_pt_trip_id)) in unmatched_pairs:
        return "car"

    return nominal_kind


# ============================================================
# Bike helper
# ============================================================
def incoming_pref(seg_kind: str) -> str:
    return "passenger" if seg_kind == "car" else "pedestrian"


def choose_bike_edges_and_pos(from_fac: str, to_fac: str):
    fe_p, _fl_p, fp_p = get_fac_sumo(from_fac, "pedestrian")
    te_p, _tl_p, tp_p = get_fac_sumo(to_fac, "pedestrian")
    edge_obj(fe_p)
    edge_obj(te_p)

    candidates = [((fe_p, fp_p), (te_p, tp_p), "ped,ped")]

    fe_b = te_b = None
    try:
        fe_b, _fl_b, fp_b = get_fac_sumo_strict(from_fac, "bicycle")
        edge_obj(fe_b)
    except Exception:
        fe_b = None

    try:
        te_b, _tl_b, tp_b = get_fac_sumo_strict(to_fac, "bicycle")
        edge_obj(te_b)
    except Exception:
        te_b = None

    if te_b is not None:
        candidates.append(((fe_p, fp_p), (te_b, tp_b), "ped,bike"))
    if fe_b is not None:
        candidates.append(((fe_b, fp_b), (te_p, tp_p), "bike,ped"))
    if fe_b is not None and te_b is not None:
        candidates.append(((fe_b, fp_b), (te_b, tp_b), "bike,bike"))

    for (fe, fp), (te, tp), _tag in candidates:
        if has_route(fe, te, "bicycle"):
            return fe, fp, te, tp

    reason = connectivity_reason(fe_p, te_p, "bicycle")
    if reason:
        raise ValueError(f"no bicycle route {fe_p}->{te_p} ({reason})")
    raise ValueError(f"no bicycle route {fe_p}->{te_p}")


# ============================================================
# Writer
# ============================================================
def write_person_to_string(person_elem: ET.Element) -> str:
    """
    V3 stateful writer.

    Core idea:
      - keep a current anchor (edge/lane/pos) for the person;
      - write every <stop> on that current anchor;
      - before a motorized/bike/PT departure, walk only to the edge required by
        the next mode using the facility CSV mapping;
      - after each segment, update the current anchor to the actual arrival edge.

    This avoids the slow candidate-pair search and prevents discontinuities like:
      walk A->B, then stop on C.
    """
    pid = person_elem.get("id", "")
    plan = get_selected_plan(person_elem)
    if plan is None:
        raise ValueError("no plan")

    acts, original_segments = build_activity_leg_sequence(plan)
    if not acts:
        raise ValueError("no non-interaction activities")

    acts, merged_segments = merge_same_facility_preserve_trip_ids(acts, original_segments)

    seg_kinds = []
    for seg in merged_segments:
        legs = seg["legs"]
        car_role = classify_car_role(legs)
        if car_role is not None:
            seg_kinds.append(("car", car_role))
            continue
        if has_any_pt(legs):
            seg_kinds.append(("pt", None))
            continue
        if has_any_bike(legs):
            seg_kinds.append(("bike", None))
            continue
        seg_kinds.append(("walk", None))

    # V3: no global precheck by default. The writing phase checks only transitions
    # that are actually needed from the real current position.
    if ENABLE_CONNECTIVITY_PRECHECK:
        for i in range(len(acts) - 1):
            fac_a = acts[i].get("facility")
            fac_b = acts[i + 1].get("facility")
            kind, _extra = seg_kinds[i]
            if not fac_a or not fac_b:
                continue
            if kind == "car":
                ea, _, _ = get_fac_sumo(fac_a, "passenger")
                eb, _, _ = get_fac_sumo(fac_b, "passenger")
                edge_obj(ea); edge_obj(eb)
                if not has_route(ea, eb, "passenger"):
                    reason = connectivity_reason(ea, eb, "passenger")
                    raise ValueError(f"no car route {ea}->{eb} ({reason})")
            elif kind == "bike":
                _ = choose_bike_edges_and_pos(fac_a, fac_b)
            # For walk/PT, stateful writing handles the real current edge and
            # candidate fallback, so prechecking primary->primary would be wrong.

    buf = io.StringIO()
    buf.write(f'    <person id="{pid}" depart="0">\n')

    current_edge = current_lane = None
    current_pos = None

    for i, act in enumerate(acts):
        act_type = act.get("type") or "unknown"
        facility = act.get("facility")
        if not facility:
            raise ValueError("activity missing facility")

        end_time = act.get("end_time")
        until_sec = hms_to_seconds(end_time) if end_time else 86400
        if FORCE_LAST_STOP_TO_86400 and i == len(acts) - 1:
            until_sec = 86400

        # First activity: choose a sane initial anchor. After that, the stop is
        # written where the previous segment really arrived.
        if current_edge is None:
            if i == 0:
                arr_kind = seg_kinds[0][0] if seg_kinds else "walk"
            else:
                prev_nominal_kind = seg_kinds[i - 1][0]
                arr_kind = effective_segment_kind_for_arrival(pid, merged_segments[i - 1], prev_nominal_kind)
            current_edge, current_lane, current_pos = get_fac_sumo(facility, incoming_pref(arr_kind))

        edge_obj(current_edge)

        buf.write(
            f'        <stop lane="{current_lane}" startPos="{fmt_pos(current_pos)}" '
            f'until="{until_sec}" actType="{act_type}" />\n'
        )

        if i == len(acts) - 1:
            break

        next_fac = acts[i + 1].get("facility")
        if not next_fac:
            raise ValueError("next activity missing facility")

        kind, extra = seg_kinds[i]
        merged_seg = merged_segments[i]

        # ---------------- WALK ----------------
        if kind == "walk":
            # Only choose a reachable destination anchor for the next activity.
            dst_edge, dst_lane, dst_pos = choose_pedestrian_target_from_current(
                current_edge=current_edge,
                current_pos=current_pos,
                to_fac=next_fac,
            )

            if current_edge != dst_edge or abs(float(current_pos) - float(dst_pos)) > 1e-6:
                # Same-edge walks do not need a graph search.
                if current_edge == dst_edge or has_route(current_edge, dst_edge, "pedestrian"):
                    buf.write(
                        f'        <walk from="{current_edge}" to="{dst_edge}" '
                        f'arrivalPos="{fmt_pos(dst_pos)}" />\n'
                    )
                else:
                    reason = connectivity_reason(current_edge, dst_edge, "pedestrian")
                    raise ValueError(f"no pedestrian route {current_edge}->{dst_edge} ({reason})")

            current_edge, current_lane, current_pos = dst_edge, dst_lane, dst_pos

        # ---------------- CAR ----------------
        elif kind == "car":
            car_role = extra or "car"

            # Vehicle departure/arrival anchors come directly from facilities CSV.
            dep_edge, dep_lane, dep_pos = get_fac_sumo(facility, "passenger")
            dst_edge, dst_lane, dst_pos = get_fac_sumo(next_fac, "passenger")
            edge_obj(dep_edge); edge_obj(dst_edge)

            # If the current stop is on a pedestrian edge of the same POI, walk to
            # the passenger edge before starting the car. This is the user's idea.
            dep_edge, dep_pos = connect_activity_to_vehicle_start(
                buf=buf,
                from_edge=current_edge,
                from_pos=current_pos,
                target_edge=dep_edge,
                target_pos=dep_pos,
                target_vclass="passenger",
                context=f"car person={pid} facility={facility}",
            )
            dep_lane = dep_lane if dep_edge == get_fac_sumo(facility, "passenger")[0] else current_lane

            if not has_route(dep_edge, dst_edge, "passenger"):
                reason = connectivity_reason(dep_edge, dst_edge, "passenger")
                raise ValueError(f"no car route {dep_edge}->{dst_edge} ({reason})")

            buf.write(
                f'        <personTrip from="{dep_edge}" to="{dst_edge}" '
                f'departPos="{fmt_pos(dep_pos)}" arrivalPos="{fmt_pos(dst_pos)}" '
                f'modes="car" role="{car_role}" />\n'
            )

            current_edge, current_lane, current_pos = dst_edge, dst_lane, dst_pos

        # ---------------- BIKE ----------------
        elif kind == "bike":
            dep_edge, dep_pos, dst_edge, dst_pos = choose_bike_edges_and_pos(facility, next_fac)
            edge_obj(dep_edge); edge_obj(dst_edge)

            dep_edge, dep_pos = connect_activity_to_vehicle_start(
                buf=buf,
                from_edge=current_edge,
                from_pos=current_pos,
                target_edge=dep_edge,
                target_pos=dep_pos,
                target_vclass="bicycle",
                context=f"bike person={pid} facility={facility}",
            )

            if not has_route(dep_edge, dst_edge, "bicycle"):
                reason = connectivity_reason(dep_edge, dst_edge, "bicycle")
                raise ValueError(f"no bicycle route {dep_edge}->{dst_edge} ({reason})")

            # Recover a lane for the destination anchor: prefer strict bicycle,
            # then pedestrian, then any facility mapping.
            try:
                _dst_e2, dst_lane, _dst_pos2 = get_fac_sumo_strict(next_fac, "bicycle")
                if _dst_e2 != dst_edge:
                    raise KeyError
            except Exception:
                dst_edge_tmp, dst_lane, dst_pos_tmp = get_fac_sumo(next_fac, "pedestrian")
                if dst_edge_tmp == dst_edge:
                    dst_pos = dst_pos_tmp
                else:
                    dst_lane = first_lane_for_vclass(dst_edge, "bicycle")

            buf.write(
                f'        <personTrip from="{dep_edge}" to="{dst_edge}" '
                f'departPos="{fmt_pos(dep_pos)}" arrivalPos="{fmt_pos(dst_pos)}" '
                f'modes="bicycle" />\n'
            )

            current_edge, current_lane, current_pos = dst_edge, dst_lane, dst_pos

        # ---------------- PT ----------------
        elif kind == "pt":
            act_edge_p, act_lane_p, act_pos_p = get_fac_sumo(facility, "pedestrian")
            nxt_edge_p, nxt_lane_p, nxt_pos_p = get_fac_sumo(next_fac, "pedestrian")
            edge_obj(act_edge_p); edge_obj(nxt_edge_p)

            # Move from the real current stop to the pedestrian access edge if
            # possible. If not, keep current_edge as PT origin; SUMO may still
            # route edge->busStop depending on the network.
            pt_origin_edge = current_edge
            pt_origin_pos = current_pos
            if current_edge != act_edge_p or abs(float(current_pos) - float(act_pos_p)) > 1e-6:
                if current_edge == act_edge_p or has_route(current_edge, act_edge_p, "pedestrian"):
                    buf.write(
                        f'        <walk from="{current_edge}" to="{act_edge_p}" '
                        f'arrivalPos="{fmt_pos(act_pos_p)}" />\n'
                    )
                    pt_origin_edge = act_edge_p
                    pt_origin_pos = act_pos_p
                else:
                    pt_origin_edge = current_edge
                    pt_origin_pos = current_pos

            original_pt_trip_id = resolve_original_pt_trip_id(pid, merged_seg)

            # Fallback taxi only for trips explicitly listed as unmatched.
            if (str(pid), int(original_pt_trip_id)) in unmatched_pairs:
                dep_edge, dep_lane, dep_pos = get_fac_sumo(facility, "passenger")
                dst_edge, dst_lane, dst_pos = get_fac_sumo(next_fac, "passenger")
                edge_obj(dep_edge); edge_obj(dst_edge)

                dep_edge, dep_pos = connect_activity_to_vehicle_start(
                    buf=buf,
                    from_edge=pt_origin_edge,
                    from_pos=pt_origin_pos,
                    target_edge=dep_edge,
                    target_pos=dep_pos,
                    target_vclass="passenger",
                    context=f"pt_unmatched_car person={pid} facility={facility}",
                )

                if not has_route(dep_edge, dst_edge, "passenger"):
                    reason = connectivity_reason(dep_edge, dst_edge, "passenger")
                    raise ValueError(f"no car route {dep_edge}->{dst_edge} ({reason})")

                buf.write(
                    f'        <personTrip from="{dep_edge}" to="{dst_edge}" '
                    f'departPos="{fmt_pos(dep_pos)}" arrivalPos="{fmt_pos(dst_pos)}" '
                    f'modes="car" role="car_passenger" />\n'
                )

                current_edge, current_lane, current_pos = dst_edge, dst_lane, dst_pos
                continue

            pt_chain = build_pt_chain_for_trip(pid, original_pt_trip_id)

            current_origin_type = "edge"
            current_origin_ref = pt_origin_edge
            forced_board_stop = None

            for k, leg in enumerate(pt_chain):
                line = leg["sumo_line"]

                # BOARD STOP
                if forced_board_stop is not None:
                    board_idx, board_stop = find_stop_index_by_id(line, forced_board_stop["busStopId"])
                    if board_stop is None:
                        raise ValueError(
                            f'forced transfer board stop "{forced_board_stop["busStopId"]}" not found on line {line} '
                            f'for person={pid}, original_trip_id={original_pt_trip_id}, leg_index={leg["leg_index"]}'
                        )
                    board_name_used = board_stop["name"]
                    forced_board_stop = None
                else:
                    board_primary = leg["access_name_primary"]
                    board_fallbacks = leg["access_name_fallbacks"]
                    board_idx, board_stop, board_name_used = find_named_stop_after(
                        line=line,
                        primary_name=board_primary,
                        min_idx=0,
                        fallback_names=board_fallbacks,
                    )

                    if board_stop is None:
                        raise ValueError(
                            f'cannot find board stop "{board_primary or board_fallbacks}" on line {line} '
                            f'for person={pid}, original_trip_id={original_pt_trip_id}, leg_index={leg["leg_index"]}'
                        )

                if current_origin_type == "edge":
                    buf.write(
                        f'        <walk from="{current_origin_ref}" busStop="{board_stop["busStopId"]}" />\n'
                    )
                elif current_origin_type == "busStop":
                    if current_origin_ref != board_stop["busStopId"]:
                        buf.write(
                            f'        <walk fromBusStop="{current_origin_ref}" busStop="{board_stop["busStopId"]}" />\n'
                        )
                else:
                    raise ValueError(f"unknown PT origin type: {current_origin_type}")

                # LAST PT LEG => alight to destination
                if k == len(pt_chain) - 1:
                    alight_primary = leg["egress_name_primary"]
                    alight_fallbacks = leg["egress_name_fallbacks"]

                    alight_idx, alight_stop, alight_name_used = find_named_stop_after(
                        line=line,
                        primary_name=alight_primary,
                        min_idx=board_idx + 1,
                        fallback_names=alight_fallbacks,
                    )

                    if alight_stop is None:
                        raise ValueError(
                            f'cannot find alight stop "{alight_primary or alight_fallbacks}" after "{board_name_used}" on line {line} '
                            f'for person={pid}, original_trip_id={original_pt_trip_id}, leg_index={leg["leg_index"]}'
                        )

                    buf.write(
                        f'        <ride fromBusStop="{board_stop["busStopId"]}" '
                        f'busStop="{alight_stop["busStopId"]}" '
                        f'lines="{line}" />\n'
                    )
                    buf.write(
                        f'        <walk fromBusStop="{alight_stop["busStopId"]}" to="{nxt_edge_p}" '
                        f'arrivalPos="{fmt_pos(nxt_pos_p)}" />\n'
                    )

                    current_edge, current_lane, current_pos = nxt_edge_p, nxt_lane_p, nxt_pos_p

                # TRANSFER TO NEXT PT LEG
                else:
                    next_leg = pt_chain[k + 1]

                    transfer_name_candidates = unique_nonempty(
                        [leg["egress_name_primary"]]
                        + leg["egress_name_fallbacks"]
                        + [next_leg["access_name_primary"]]
                        + next_leg["access_name_fallbacks"]
                    )

                    transfer_found = False
                    transfer_alight = None
                    transfer_board_next = None

                    for nm in transfer_name_candidates:
                        a_idx, a_stop, _ = find_named_stop_after(
                            line=line,
                            primary_name=nm,
                            min_idx=board_idx + 1,
                            fallback_names=[]
                        )
                        b_idx2, b_stop2, _ = find_named_stop_after(
                            line=next_leg["sumo_line"],
                            primary_name=nm,
                            min_idx=0,
                            fallback_names=[]
                        )
                        if a_stop is not None and b_stop2 is not None:
                            transfer_alight = a_stop
                            transfer_board_next = b_stop2
                            transfer_found = True
                            break

                    if not transfer_found:
                        (a_idx, a_stop), (b_idx2, b_stop2) = first_common_transfer(
                            line1=line,
                            min_idx1=board_idx + 1,
                            line2=next_leg["sumo_line"]
                        )
                        if a_stop is None or b_stop2 is None:
                            raise ValueError(
                                f"no transfer stop found between {line} and {next_leg['sumo_line']} "
                                f"for person={pid}, original_trip_id={original_pt_trip_id}"
                            )
                        transfer_alight = a_stop
                        transfer_board_next = b_stop2

                    buf.write(
                        f'        <ride fromBusStop="{board_stop["busStopId"]}" '
                        f'busStop="{transfer_alight["busStopId"]}" '
                        f'lines="{line}" />\n'
                    )

                    current_origin_type = "busStop"
                    current_origin_ref = transfer_alight["busStopId"]
                    forced_board_stop = transfer_board_next

        else:
            raise ValueError(f"unknown segment kind: {kind}")

    buf.write("    </person>\n")
    return buf.getvalue()


# ============================================================
# RUN
# ============================================================
total = written = skipped = 0
skipped_nan = skipped_no_route = skipped_unknown_edge = skipped_pt = 0

with OUT_LOG.open("w", newline="", encoding="utf-8") as log_f, OUT_ROU.open("w", encoding="utf-8") as out_f:
    log = csv.writer(log_f)
    log.writerow(["person_id", "status", "reason"])

    out_f.write('<?xml version="1.0" encoding="UTF-8"?>\n<routes>\n')

    with zstd.open(POP_PATH, "rb") as f:
        for event, elem in ET.iterparse(f, events=("end",)):
            if ln(elem.tag) == "person":
                pid = elem.get("id", "")

                if TEST_IDS is not None and pid not in TEST_IDS:
                    elem.clear()
                    continue

                total += 1

                if TEST_MAX_PERSONS is not None and total > TEST_MAX_PERSONS:
                    elem.clear()
                    break

                try:
                    out_f.write(write_person_to_string(elem))
                    written += 1
                    log.writerow([pid, "ok", ""])

                except Exception as e:
                    skipped += 1
                    msg = str(e).lower()

                    if "nan mapping" in msg or "nan" in msg:
                        skipped_nan += 1
                        log.writerow([pid, "skipped_nan", str(e)])

                    elif ("no pedestrian route" in msg) or ("no car route" in msg) or ("no bicycle route" in msg) \
                         or ("no pedestrian access" in msg) or ("edge not allowed" in msg):
                        skipped_no_route += 1
                        log.writerow([pid, "skipped_no_route", str(e)])

                    elif ("cannot find board stop" in msg) or ("cannot find alight stop" in msg) \
                         or ("no transfer stop found" in msg) or ("no pt mapping rows" in msg) \
                         or ("missing access stop name" in msg) or ("missing egress stop name" in msg) \
                         or ("mapped sumo line not present" in msg) \
                         or ("multiple pt mapping rows candidates" in msg) \
                         or ("multiple unmatched pt trips candidates" in msg) \
                         or ("multiple original pt trips merged" in msg) \
                         or ("merged pt segment has no original pt trip" in msg) \
                         or ("forced transfer board stop" in msg):
                        skipped_pt += 1
                        log.writerow([pid, "skipped_pt", str(e)])

                    elif "unknown edge in net" in msg:
                        skipped_unknown_edge += 1
                        log.writerow([pid, "skipped_unknown_edge", str(e)])

                    else:
                        log.writerow([pid, "skipped_other", str(e)])

                elem.clear()

                if total % PRINT_EVERY == 0:
                    print(
                        f"[progress] total={total} written={written} skipped={skipped} "
                        f"(nan={skipped_nan}, noroute={skipped_no_route}, pt={skipped_pt})"
                    )

    out_f.write("</routes>\n")

print("\n[OK] DONE")
print(f"total={total}, written={written}, skipped={skipped}")
print(f"  - skipped_nan         : {skipped_nan}")
print(f"  - skipped_no_route    : {skipped_no_route}")
print(f"  - skipped_unknown_edge: {skipped_unknown_edge}")
print(f"  - skipped_pt          : {skipped_pt}")
print(f"Routes: {OUT_ROU.resolve()}")
print(f"Log   : {OUT_LOG.resolve()}")
print(f"PT mapped rows: {len(pt_join)}")
print(f"PT unmatched trips: {len(unmatched_pairs)}")
print(f"SUMO PT lines : {len(available_lines)}")
print(f"Shortest-path cache entries: {len(_sp_cost_cache)}")
print(f"Shortest-path cache hits/misses: {_route_cache_hits}/{_route_cache_misses}")
save_route_cache()


[INFO] Loading SUMO network...
[INFO] Network file: ..\1-network\cda_la_rochelle.net.xml
[INFO] withPedestrianConnections=True
[INFO] Pedestrian candidate fallback loaded: 15614 POIs
[progress] total=100 written=96 skipped=4 (nan=0, noroute=4, pt=0)
[progress] total=200 written=191 skipped=9 (nan=0, noroute=9, pt=0)
[progress] total=300 written=285 skipped=15 (nan=0, noroute=13, pt=2)
[progress] total=400 written=381 skipped=19 (nan=0, noroute=16, pt=3)
[progress] total=500 written=481 skipped=19 (nan=0, noroute=16, pt=3)
[progress] total=600 written=575 skipped=25 (nan=0, noroute=21, pt=3)
[progress] total=700 written=672 skipped=28 (nan=0, noroute=22, pt=5)
[progress] total=800 written=772 skipped=28 (nan=0, noroute=22, pt=5)
[progress] total=900 written=870 skipped=30 (nan=0, noroute=24, pt=5)
[progress] total=1000 written=969 skipped=31 (nan=0, noroute=25, pt=5)
[progress] total=1100 written=1065 skipped=35 (nan=0, noroute=28, pt=6)
[progress] total=1200 written=1163 skipped=37 (na